In [ ]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# =========================
# Models
# =========================


class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 64))
        self.dec = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 28 * 28), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.enc(x)
        x_hat = self.dec(z)
        return x_hat, z


class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc_mu = nn.Linear(256, 64)
        self.fc_logvar = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 256)
        self.fc3 = nn.Linear(256, 28 * 28)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))

    def forward(self, x):
        x = x.view(x.size(0), -1)
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


class VectorQuantizer(nn.Module):
    def __init__(self, K=512, D=64, beta=0.25):
        super().__init__()
        self.embedding = nn.Embedding(K, D)
        self.embedding.weight.data.uniform_(-1 / K, 1 / K)
        self.beta = beta

    def forward(self, z):
        flat = z.view(-1, z.size(-1))

        dist = (
            flat.pow(2).sum(1, keepdim=True)
            - 2 * flat @ self.embedding.weight.t()
            + self.embedding.weight.pow(2).sum(1)
        )

        idx = torch.argmin(dist, dim=1)
        z_q = self.embedding(idx).view_as(z)

        loss = F.mse_loss(z_q.detach(), z) + self.beta * F.mse_loss(z_q, z.detach())
        z_q = z + (z_q - z).detach()

        return z_q, loss


class VQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(28 * 28, 256), nn.ReLU(), nn.Linear(256, 64))
        self.vq = VectorQuantizer()
        self.dec = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 28 * 28), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.enc(x)
        z_q, vq_loss = self.vq(z)
        x_hat = self.dec(z_q)
        return x_hat, vq_loss


# =========================
# Train
# =========================


def train(model, loader, device, model_type):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(3):
        for x, _ in loader:
            x = x.to(device)

            if model_type == "ae":
                x_hat, _ = model(x)
                loss = F.mse_loss(x_hat, x.view(x.size(0), -1))

            elif model_type == "vae":
                x_hat, mu, logvar = model(x)
                recon = F.mse_loss(x_hat, x.view(x.size(0), -1))
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                loss = recon + kl

            elif model_type == "vqvae":
                x_hat, vq_loss = model(x)
                recon = F.mse_loss(x_hat, x.view(x.size(0), -1))
                loss = recon + vq_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

        print(f"epoch {epoch} loss {loss.item():.4f}")


# =========================
# Main
# =========================


def main(model_type="ae"):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    ds = datasets.MNIST(
        "./data", train=True, download=True, transform=transforms.ToTensor()
    )
    loader = DataLoader(ds, batch_size=128, shuffle=True)

    if model_type == "ae":
        model = AE().to(device)
    elif model_type == "vae":
        model = VAE().to(device)
    elif model_type == "vqvae":
        model = VQVAE().to(device)

    train(model, loader, device, model_type)


# 実行
main("ae")
